# Topico 1: Entropia, Informacao Mutua e o Teste de Independencia

Cenario: teste rapido de dengue. D = estado real, T = resultado do teste. Enunciado: `propostas/1_Simulacoes.pdf`, secao 1.

## Preparacao

In [ ]:
import sys, pathlib

REPO = pathlib.Path.cwd().parents[2]          # .../<topico>/ -> .../<proposta>/ -> raiz
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2

from utils.info_theory import (
    entropy, joint_entropy, conditional_entropy,
    cross_entropy, kl_divergence, mutual_information,
    mi_gtest_statistic, counts_to_joint,
)
from utils.simulation import (
    rng, empirical_pmf, markov_chain, stationary_distribution,
)
from utils.estimation import bootstrap_ci
from utils.plotting import (
    setup_estilo, salvar_fig, plot_barras, tabela, tabela_conjunta, tabela_comparacao,
    plot_estimativa_vs_n, plot_curvas,
)
import src.model as model                     # helpers especificos deste topico

setup_estilo()
GEN = rng(20260902)                           # seed fixo -> resultados reproduziveis
FIGURAS = pathlib.Path("figuras")

## Passo 1: Modelo verdadeiro

Definir P(D=1), P(T=1|D=0) e P(T=0|D=1) (`model.true_joint`). Calcular os valores TEORICOS de H(D), H(T), H(D,T), H(D|T) e I(D;T).

In [ ]:
# Passo 1
PREVALENCIA = 0.08   # P(D = 1)
FPR = 0.03           # P(T = 1 | D = 0), falso positivo
FNR = 0.12           # P(T = 0 | D = 1), falso negativo

JOINT_TEORICO = model.true_joint(PREVALENCIA, FPR, FNR)
display(tabela_conjunta(JOINT_TEORICO, rotulos_x=["D=0", "D=1"], rotulos_y=["T=0", "T=1"]))

P_D_TEORICO = JOINT_TEORICO.sum(axis=1)
P_T_TEORICO = JOINT_TEORICO.sum(axis=0)

TEORICO = {
    "H(D)": entropy(P_D_TEORICO),
    "H(T)": entropy(P_T_TEORICO),
    "H(D,T)": joint_entropy(JOINT_TEORICO),
    "H(D|T)": conditional_entropy(JOINT_TEORICO, given="y"),  # H(X | Y) = H(D | T)
    "I(D;T)": mutual_information(JOINT_TEORICO),
}
for nome, valor in TEORICO.items():
    print(f"{nome} = {valor:.4f} bits")

plot_barras(TEORICO, ylabel="bits")

## Passo 2: Amostra de N pares (D, T)

Gerar N pacientes com `model.sample_pairs` (sorteia D pela marginal, T condicional a D).

In [ ]:
# Passo 2
N = 2000  # tamanho da amostra "N fixo" usada na comparacao do Passo 3

D_AMOSTRA, T_AMOSTRA = model.sample_pairs(JOINT_TEORICO, N, GEN)
print(f"N = {N} pares (D, T) gerados")
print(f"prevalencia empirica de D: {D_AMOSTRA.mean():.4f}  (teorica: {PREVALENCIA})")
print(f"positividade empirica de T: {T_AMOSTRA.mean():.4f}  (teorica: {P_T_TEORICO[1]:.4f})")

## Passo 3: Estimativas na amostra

Montar a tabela conjunta empirica (`counts_to_joint`) e reestimar H(D), H(T), H(D,T), H(D|T), I(D;T). Comparar com o Passo 1.

In [ ]:
# Passo 3
def _i_dt(d, t):
    """I(D;T) a partir de dois vetores de amostras (usado tambem no Passo 4)."""
    return mutual_information(counts_to_joint(d, t, size_x=2, size_y=2))


JOINT_AMOSTRA = counts_to_joint(D_AMOSTRA, T_AMOSTRA, size_x=2, size_y=2)
display(tabela_conjunta(JOINT_AMOSTRA, rotulos_x=["D=0", "D=1"], rotulos_y=["T=0", "T=1"], normalizar=True))

P_D_AMOSTRA = JOINT_AMOSTRA.sum(axis=1)
P_T_AMOSTRA = JOINT_AMOSTRA.sum(axis=0)

ESTIMADO = {
    "H(D)": entropy(P_D_AMOSTRA),
    "H(T)": entropy(P_T_AMOSTRA),
    "H(D,T)": joint_entropy(JOINT_AMOSTRA),
    "H(D|T)": conditional_entropy(JOINT_AMOSTRA, given="y"),
    "I(D;T)": mutual_information(JOINT_AMOSTRA),
}

display(tabela_comparacao(list(TEORICO), list(ESTIMADO.values()), list(TEORICO.values())))

# Incerteza de I(D;T) nesta amostra unica, por reamostragem dos proprios N pares.
I_EST, I_LO, I_HI = bootstrap_ci((D_AMOSTRA, T_AMOSTRA), _i_dt, generator=GEN, paired=True)
print(f"I(D;T) = {I_EST:.4f} bits, IC bootstrap 95% = [{I_LO:.4f}, {I_HI:.4f}]")
print(f"o IC cobre o valor teorico ({TEORICO['I(D;T)']:.4f})? {bool(I_LO <= TEORICO['I(D;T)'] <= I_HI)}")

## Passo 4: Erro de I(D;T) vs N

Repetir a estimativa de I para N = 20, 50, 100, 500, 2000, 10000 e observar o erro vs o valor teorico. Sao R amostras independentes por N (nao uma so), para separar o vies sistematico do acaso de uma tirada.

In [ ]:
# Passo 4
# Uma amostra por N daria uma curva de erro dominada pelo acaso daquela tirada:
# em N = 20 o erro relativo de uma realizacao qualquer vai de ~19% a ~100%.
# Repetindo R vezes por N, o que a curva mostra e o comportamento SISTEMATICO,
# inclusive o vies para cima que o enunciado avisa existir em amostra pequena.
NS = [20, 50, 100, 500, 2000, 10000]
R_PASSO4 = 400  # repeticoes independentes por N

I_MEDIO, I_P10, I_P90 = [], [], []
for n in NS:
    amostras_i = np.array(
        [_i_dt(*model.sample_pairs(JOINT_TEORICO, n, GEN)) for _ in range(R_PASSO4)]
    )
    I_MEDIO.append(float(amostras_i.mean()))
    I_P10.append(float(np.percentile(amostras_i, 10)))
    I_P90.append(float(np.percentile(amostras_i, 90)))

display(tabela_comparacao(
    [f"N={n}" for n in NS], I_MEDIO, [TEORICO["I(D;T)"]] * len(NS), I_P10, I_P90,
))

plot_estimativa_vs_n(
    NS, I_MEDIO, I_P10, I_P90,
    teorico=TEORICO["I(D;T)"],
    label=f"I(D;T) medio em {R_PASSO4} amostras", ylabel="bits",
)
salvar_fig("t1_passo4_erro_i_vs_n")

## Passo 5: Entropia cruzada e divergencia

Escolher q(D) != p(D), calcular H(p, q) e D(p||q). Conferir que o excesso de custo = D(p||q).

In [ ]:
# Passo 5
Q_D = np.array([0.85, 0.15])  # q(D) adivinhado, diferente de p(D) = [1 - PREVALENCIA, PREVALENCIA]

H_PQ = cross_entropy(P_D_TEORICO, Q_D)
KL_PQ = kl_divergence(P_D_TEORICO, Q_D)
EXCESSO = H_PQ - TEORICO["H(D)"]

print(f"p(D)          = {P_D_TEORICO}")
print(f"q(D)          = {Q_D}")
print(f"H(p, q)       = {H_PQ:.4f} bits")
print(f"H(p)          = {TEORICO['H(D)']:.4f} bits")
print(f"H(p,q) - H(p) = {EXCESSO:.4f} bits")
print(f"D(p||q)       = {KL_PQ:.4f} bits")
assert np.isclose(EXCESSO, KL_PQ), "H(p,q) - H(p) deveria bater com D(p||q)"

plot_barras({"H(p)": TEORICO["H(D)"], "H(p,q)": H_PQ, "D(p||q)": KL_PQ}, ylabel="bits")

## Passo 6: Dimensao temporal (Markov)

Simular o historico T_1..T_n de um paciente com uma cadeia de Markov de 2 estados (`markov_chain`), iniciada na distribuicao estacionaria (`stationary_distribution`) e calibrada para ter a mesma prevalencia do Passo 1.

In [ ]:
# Passo 6
# A cadeia e escolhida para ter a MESMA prevalencia do Passo 1 em regime, senao
# a extensao temporal descreveria outra populacao e os dois blocos do notebook
# deixariam de ser comparaveis. Com P(D=0->1) = a e P(D=1->0) = b, a
# estacionaria e [b, a] / (a + b); logo b = a (1 - prevalencia) / prevalencia.
A_INFECTA = 0.02                                       # P(D=0 -> D=1) no dia seguinte
B_CURA = A_INFECTA * (1 - PREVALENCIA) / PREVALENCIA   # P(D=1 -> D=0) no dia seguinte
P_D_MARKOV = np.array([
    [1 - A_INFECTA, A_INFECTA],
    [B_CURA, 1 - B_CURA],
])
PI_D = stationary_distribution(P_D_MARKOV)
N_DIAS = 14

print(f"matriz de transicao de D:\n{P_D_MARKOV}")
print(f"estacionaria PI_D = {PI_D}  (P(D=1) = {PI_D[1]:.4f}, prevalencia do Passo 1 = {PREVALENCIA})")

# x0 sorteado da estacionaria: a cadeia ja comeca em regime, entao T_1 tem a
# mesma distribuicao de T_8 e o baseline n * H(T_1) do Passo 8 e uma comparacao
# justa. Com x0 fixo ou uniforme, H(T_1) refletiria a condicao inicial.
D_SERIE = markov_chain(P_D_MARKOV, N_DIAS, GEN, x0=int(GEN.choice(2, p=PI_D)))
T_SERIE = model.sample_given_d(JOINT_TEORICO, D_SERIE, GEN)

print("D_1..D_n:", D_SERIE)
print("T_1..T_n:", T_SERIE)

SERIE_DF = pd.DataFrame({
    "dia": np.tile(np.arange(1, N_DIAS + 1), 2),
    "valor": np.concatenate([D_SERIE, T_SERIE]),
    "variavel": ["D"] * N_DIAS + ["T"] * N_DIAS,
})
plot_curvas(SERIE_DF, x="dia", y="valor", hue="variavel", xlabel="dia", ylabel="estado (0/1)")
salvar_fig("t1_passo6_serie_d_t")

## Passo 7: Regra da cadeia

Estimar H(T_1..T_n) por contagem; comparar com a soma das entropias condicionais de ordem 1 e com n*H(T_1). Conferir os tres contra o valor exato, que sai do algoritmo forward porque (D, T) e um modelo oculto de Markov.

In [ ]:
# Passo 7
N_CADEIA = 8          # dias por paciente nesta estimativa (2^8 = 256 sequencias possiveis)
M_PACIENTES = 20000   # repeticoes independentes de markov_chain + sample_given_d

T_SEQS = np.empty((M_PACIENTES, N_CADEIA), dtype=int)
for m in range(M_PACIENTES):
    d_m = markov_chain(P_D_MARKOV, N_CADEIA, GEN, x0=int(GEN.choice(2, p=PI_D)))
    T_SEQS[m] = model.sample_given_d(JOINT_TEORICO, d_m, GEN)

# --- estimativas por contagem -------------------------------------------------
# H(T_1,...,T_n) por contagem direta sobre o alfabeto 2^n (cada sequencia -> um inteiro)
POTENCIAS_DE_2 = 2 ** np.arange(N_CADEIA - 1, -1, -1)
CODIGOS_SEQ = T_SEQS @ POTENCIAS_DE_2
H_SEQ_ESTIMADO = entropy(empirical_pmf(CODIGOS_SEQ, size=2**N_CADEIA))

H_T1 = entropy(empirical_pmf(T_SEQS[:, 0], size=2))
SOMA_ORDEM1 = H_T1
for i in range(1, N_CADEIA):
    joint_i = counts_to_joint(T_SEQS[:, i - 1], T_SEQS[:, i], size_x=2, size_y=2)
    SOMA_ORDEM1 += conditional_entropy(joint_i, given="x")  # H(T_i | T_(i-1))
N_VEZES_H_T1 = N_CADEIA * H_T1

# --- os mesmos tres valores, exatos -------------------------------------------
# D e Markov e T depende so do D do mesmo dia: (D, T) e um modelo oculto de
# Markov, entao p(t_1..t_n) sai exata pelo algoritmo forward e serve de teorico.
PMF_SEQ_EXATA = model.t_sequence_pmf(JOINT_TEORICO, P_D_MARKOV, N_CADEIA, PI_D)
H_SEQ_EXATO = entropy(PMF_SEQ_EXATA, normalize=False)
MARG_EXATAS = model.t_marginals(JOINT_TEORICO, P_D_MARKOV, N_CADEIA, PI_D)
PARES_EXATOS = model.t_pair_joints(JOINT_TEORICO, P_D_MARKOV, N_CADEIA, PI_D)
H_T1_EXATO = entropy(MARG_EXATAS[0])
SOMA_ORDEM1_EXATA = H_T1_EXATO + sum(conditional_entropy(p, given="x") for p in PARES_EXATOS)
N_VEZES_H_T1_EXATO = N_CADEIA * H_T1_EXATO

display(tabela_comparacao(
    ["H(T_1..T_n)", "soma H(T_i|T_i-1)", "n x H(T_1)"],
    [H_SEQ_ESTIMADO, SOMA_ORDEM1, N_VEZES_H_T1],
    [H_SEQ_EXATO, SOMA_ORDEM1_EXATA, N_VEZES_H_T1_EXATO],
))

# CUIDADO: T sozinho NAO e uma cadeia de Markov (quem e Markov e D; T e a
# observacao ruidosa dele). A regra da cadeia condiciona em TODO o passado,
# H(T^n) = soma H(T_i | T_1..T_i-1); condicionar so em T_(i-1) e uma
# aproximacao de ordem 1, e sempre um LIMITE SUPERIOR de H(T^n).
FOLGA_ORDEM1 = SOMA_ORDEM1_EXATA - H_SEQ_EXATO
print(f"H(T_1..T_{N_CADEIA}) exata                 = {H_SEQ_EXATO:.4f} bits")
print(f"soma H(T_i|T_(i-1)) exata             = {SOMA_ORDEM1_EXATA:.4f} bits  (limite superior)")
print(f"folga da aproximacao de ordem 1       = {FOLGA_ORDEM1:.4f} bits "
      f"({100 * FOLGA_ORDEM1 / H_SEQ_EXATO:.1f}% de H(T_1..T_n))")
print(f"{N_CADEIA} x H(T_1) exato                     = {N_VEZES_H_T1_EXATO:.4f} bits  (baseline i.i.d.)")

plot_barras(
    {
        "H(T_1..T_n)": H_SEQ_EXATO,
        "soma H(T_i|T_i-1)": SOMA_ORDEM1_EXATA,
        "n*H(T_1)": N_VEZES_H_T1_EXATO,
    },
    ylabel="bits",
)
salvar_fig("t1_passo7_regra_cadeia")

## Passo 8: Economia de bits

Quantificar em bits a economia de transmitir o historico de forma condicional em vez de dia a dia isolado.

In [ ]:
# Passo 8
# A cadeia esta em regime (Passo 6), entao T_1 tem a mesma distribuicao de
# qualquer outro dia e n * H(T_1) e de fato o custo de transmitir cada dia como
# se fosse uma amostra independente da MESMA distribuicao: a comparacao mede
# dependencia temporal, e nao a condicao inicial da cadeia.
ECONOMIA_ORDEM1 = N_VEZES_H_T1_EXATO - SOMA_ORDEM1_EXATA  # olhando so o dia anterior
ECONOMIA_TOTAL = N_VEZES_H_T1_EXATO - H_SEQ_EXATO         # explorando todo o passado

print(f"custo dia a dia isolado, {N_CADEIA} x H(T_1)      = {N_VEZES_H_T1_EXATO:.4f} bits")
print(f"custo condicionando so no dia anterior   = {SOMA_ORDEM1_EXATA:.4f} bits")
print(f"custo condicionando em todo o passado    = {H_SEQ_EXATO:.4f} bits")
print()
print(f"economia com memoria de 1 dia   = {ECONOMIA_ORDEM1:.4f} bits "
      f"({100 * ECONOMIA_ORDEM1 / N_VEZES_H_T1_EXATO:.1f}% do custo isolado)")
print(f"economia com memoria completa   = {ECONOMIA_TOTAL:.4f} bits "
      f"({100 * ECONOMIA_TOTAL / N_VEZES_H_T1_EXATO:.1f}% do custo isolado)")
print(f"-> a memoria de 1 dia ja captura {100 * ECONOMIA_ORDEM1 / ECONOMIA_TOTAL:.1f}% da economia possivel")

plot_barras(
    {
        "n*H(T_1) (isolado)": N_VEZES_H_T1_EXATO,
        "so o dia anterior": SOMA_ORDEM1_EXATA,
        "todo o passado": H_SEQ_EXATO,
    },
    ylabel="bits",
)
salvar_fig("t1_passo8_economia_bits")

## Passo 9: Teste de independencia (teste-G)

Aplicar `mi_gtest_statistic` a uma amostra; reportar G, g.l. e p-valor (`scipy.stats.chi2.sf`) para D vs T e para T_2 vs T_1 (um par por paciente, para nao violar a independencia das observacoes).

In [ ]:
# Passo 9
# D x T: amostra de N pares do Passo 2/3 (contagens brutas, nao normalizadas)
CONTAGEM_D_T = JOINT_AMOSTRA

# T_i x T_(i-1): UM par por paciente. Empilhar os 7 pares consecutivos de cada
# paciente daria N = 7 x M "observacoes" que se sobrepoem duas a duas e vem da
# mesma trajetoria -- nao sao independentes. Como G escala linearmente com N,
# a pseudo-replicacao infla a estatistica (aqui, por um fator ~7) e o p-valor
# deixa de ter o significado que a distribuicao qui-quadrado lhe da.
CONTAGEM_T_LAG = counts_to_joint(T_SEQS[:, 0], T_SEQS[:, 1], size_x=2, size_y=2)

G_DT, DOF_DT, I_DT_HAT = mi_gtest_statistic(CONTAGEM_D_T)
P_DT = chi2.sf(G_DT, DOF_DT)

G_TLAG, DOF_TLAG, I_TLAG_HAT = mi_gtest_statistic(CONTAGEM_T_LAG)
P_TLAG = chi2.sf(G_TLAG, DOF_TLAG)

display(tabela(
    [
        {
            "teste": "D x T",
            "N": int(CONTAGEM_D_T.sum()),
            "I_hat (bits)": I_DT_HAT,
            "G": G_DT,
            "g.l.": DOF_DT,
            "p-valor": P_DT,
        },
        {
            "teste": "T_2 x T_1",
            "N": int(CONTAGEM_T_LAG.sum()),
            "I_hat (bits)": I_TLAG_HAT,
            "G": G_TLAG,
            "g.l.": DOF_TLAG,
            "p-valor": P_TLAG,
        },
    ],
    indice="teste",
))

for nome, g, dof, p in [("D x T", G_DT, DOF_DT, P_DT), ("T_2 x T_1", G_TLAG, DOF_TLAG, P_TLAG)]:
    rejeita = "rejeita H0 (dependentes)" if p < 0.05 else "nao rejeita H0"
    print(f"{nome}: G={g:.4f}, g.l.={dof}, p-valor={p:.3e} -> {rejeita}")

## Passo 10: Taxa de erro do teste

Repetir o experimento em 3 cenarios (dependencia forte, fraca, independencia real) e medir com que frequencia o teste acerta a decisao; taxa de rejeicao sob H0 verdadeira.

In [ ]:
# Passo 10
# Tres cenarios de D x T, variando so fpr/fnr (mesma PREVALENCIA):
# dependencia forte = Passo 1; fraca = teste quase sem poder discriminativo;
# independencia real = fpr = fnr = 0.5 (T e um lance de moeda, I(D;T) = 0 exato).
CENARIOS = {
    "dependencia forte": model.true_joint(PREVALENCIA, fpr=0.03, fnr=0.12),
    "dependencia fraca": model.true_joint(PREVALENCIA, fpr=0.35, fnr=0.40),
    "independencia real": model.true_joint(PREVALENCIA, fpr=0.50, fnr=0.50),
}

N_CENARIO = 300     # tamanho de cada amostra (D, T) dentro de uma repeticao
R_CENARIO = 1000    # repeticoes independentes do teste-G por cenario
ALPHA = 0.05

RESULTADOS_CENARIOS = []
for nome, joint in CENARIOS.items():
    rejeicoes = 0
    for _ in range(R_CENARIO):
        d_r, t_r = model.sample_pairs(joint, N_CENARIO, GEN)
        contagem = counts_to_joint(d_r, t_r, size_x=2, size_y=2)
        g, dof, _ = mi_gtest_statistic(contagem)
        if chi2.sf(g, dof) < ALPHA:
            rejeicoes += 1
    RESULTADOS_CENARIOS.append({
        "cenario": nome,
        "I(D;T) teorico (bits)": mutual_information(joint),
        "taxa de rejeicao": rejeicoes / R_CENARIO,
    })

display(tabela(RESULTADOS_CENARIOS, indice="cenario"))

for linha in RESULTADOS_CENARIOS:
    papel = "poder do teste" if linha["I(D;T) teorico (bits)"] > 0 else "taxa de erro tipo I (deveria ~ALPHA)"
    print(f"{linha['cenario']}: taxa de rejeicao = {linha['taxa de rejeicao']:.3f}  ({papel})")

plot_barras(
    {linha["cenario"]: linha["taxa de rejeicao"] for linha in RESULTADOS_CENARIOS},
    ylabel=f"taxa de rejeicao (alpha={ALPHA})",
)
salvar_fig("t1_passo10_taxa_rejeicao")


## Relatorio

Consolidar resultados, tabelas e figuras em `relatorio.md` (figuras salvas em `figuras/`).